## Figure 2

- Method Diagram
- Forecast-MMT
- Slope @ Avg Forecast

In [12]:
import sys
from pathlib import Path

PROJECT_ROOT = next(
    p for p in [Path.cwd(), *Path.cwd().parents] if (p / "utils").is_dir()
)
sys.path.insert(0, str(PROJECT_ROOT))

import os
from datetime import datetime, UTC
from dotenv import find_dotenv, load_dotenv
import xarray as xr
import xvec

import geopandas as gpd
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.ticker import MultipleLocator
import cartopy.crs as ccrs

from utils import analysis_utils
from utils import isku_utils
from utils import plotting_utils
from utils import stats_utils

import importlib

importlib.reload(analysis_utils)
importlib.reload(isku_utils)
importlib.reload(plotting_utils)

<module 'utils.plotting_utils' from '/home/emily_zuetell/projects/poreallas/utils/plotting_utils.py'>

In [13]:
load_dotenv(
    find_dotenv("october_local.env", raise_error_if_not_found=True), override=True
)

EFFECTS_URI = os.environ["POREALLAS_EFFECTS_URI"]
IMPACT_REGION_POLYGONS = os.environ["POREALLAS_REGIONS_POLYGONS_URI"]
SOCIOECONOMICS_URI = os.environ["POREALLAS_SOCIOECONOMICS_URI"]

# Climate Data
TAS_FORECAST_URI = "../data/v2610/v20261006_parsed_forecast.zarr"
ERA5_URI = os.environ["POREALLAS_ERA5_URI"]
TAS_FORECAST_RAW = "../data/v2610/v20261006_parsed_forecast.zarr"

In [14]:
# Baseline period for Impact
effect = xr.open_datatree(EFFECTS_URI, consolidated=False)
BASELINE_PERIOD = analysis_utils.get_baseline_period(effect, years=30)
# Define Forecast Months
FC_MONTHS = [10, 11, 12, 1, 2, 3]
FC_PERIOD = slice("2026-10-01", "2027-03-31")

config = analysis_utils.ImpactConfig(
    version=f"v{datetime.now(tz=UTC):%y%m%d}",
    baseline_period=BASELINE_PERIOD,
    polygons_path=os.environ["POREALLAS_REGIONS_POLYGONS_URI"],
    socioeconomics_path=os.environ["POREALLAS_SOCIOECONOMICS_URI"],
    rate=True,
    age_weight = True,
    months=FC_MONTHS,
    hotonly="hotonly",
    dims=["number", "sample"],
)

In [15]:
def bin_edges(tas_bin):
    # Standardize bin edges for discrete plotting
    tas_bin = np.asarray(tas_bin)
    return np.append(tas_bin, tas_bin[-1] + np.diff(tas_bin)[-1])


def _flat(da):
    # Flatten and check xarray
    v = np.asarray(da).ravel()
    return v[np.isfinite(v)]

In [16]:
## Climate Data Processing ##
def select_climate(forecast, reanalysis, region, pad=3):
    # Subset climate data
    fc = forecast.sel(region=region)
    era5 = reanalysis.sel(region=region)
    vals = np.concatenate([_flat(fc), _flat(era5)])
    return dict(
        forecast=fc,
        era5=era5,
        n_members=fc.sizes["number"],
        n_years=np.unique(era5.time.dt.year).size,
        xlim=(vals.min() - pad, vals.max() + pad),
    )


def plot_climate_hist(ax, clim, edges):
    # Plot climate histograms
    # Weighted for "days per month" across ERA5 years and forecast members
    for key, n, color, label in [("era5", clim["n_years"], b_color, "ERA5"),
                                 ("forecast", clim["n_members"], f_color, "Forecast")]:
        v = _flat(clim[key])
        ax.hist(v, bins=edges, weights=np.full(v.size, 1 / n),
                alpha=0.5, color=color, label=label)
    ax.set_xlim(*clim["xlim"])

In [17]:
## Response Curve ##
def select_betas(betas, mmt, region, age_cohort="age65plus"):
    # Subset response curve
    b = betas.sel(region=region, age_cohort=age_cohort).transpose("sample", "tas_bin")
    return dict(
        samples=b.values,                       # (sample, tas_bin)
        tas_bin=b["tas_bin"].values,
        mmt=mmt.sel(region=region, age_cohort=age_cohort).mean("sample").item(),
    )


def plot_response(ax, beta, ylim=(0, 10)):
    # Plot response curve and samples
    x, s = beta["tas_bin"], beta["samples"]
    # samples
    ax.step(x, s.T, where="pre", color=r_color, lw=0.5, alpha=0.15)
    # Average gamma sample
    ax.step(x, np.nanmean(s, axis=0), where="pre", color=r_color, lw=2, label="Response curve")
    # MMT
    ax.axvline(beta["mmt"], color=r_color, lw=1, label="MMT")
    ax.set_ylim(*ylim)
    ax.spines["top"].set_visible(False)

In [18]:
## CDF ##
def compute_cdf(clim, beta, q=(5, 95)):
    # CDF and Uncertainty Range
    edges = bin_edges(beta["tas_bin"])
    out = {"x": beta["tas_bin"]}
    for key, n in [("era5", clim["n_years"]), ("forecast", clim["n_members"])]:
        counts, _ = np.histogram(_flat(clim[key]), bins=edges)
        # Cumulative mortality rate = response curve * histogram / num. samples
        cdf = np.cumsum(beta["samples"] * counts / n, axis=1)   # (sample, tas_bin)
        out[key] = dict(
            mean=np.nanmean(cdf, axis=0),
            lo=np.nanpercentile(cdf, q[0], axis=0),
            hi=np.nanpercentile(cdf, q[1], axis=0),
        )
    # Log for ylim
    out["max"] = max(out["era5"]["hi"].max(), out["forecast"]["hi"].max())
    return out

def plot_cdf(ax, cdf):
# Add CDF for forecast and era5 to plot
    for key, color, label in [("forecast", f_color, "Forecast"), ("era5", b_color, "ERA5")]:
        c = cdf[key]
        ax.fill_between(cdf["x"], c["lo"], c["hi"], step="pre", color=color, alpha=0.25, lw=0)
        ax.step(cdf["x"], c["mean"], where="pre", color=color, label=label)
    ax.set_ylim(0, cdf["max"] * 1.1)

In [19]:
def plot_region_panel(fig, spec, region, title=None,
                      show_left_labels=True, show_right_labels=True):
    # Subset and compute hist and cdf
    clim = select_climate(forecast_slice, reanalysis_slice, region)
    beta = select_betas(betas, betas_mmt["mmt"], region)
    cdf = compute_cdf(clim, beta)
    # Setup gridspec
    inner = spec.subgridspec(2, 1, height_ratios=[3, 1], hspace=0.08)
    ax_hist = fig.add_subplot(inner[0])
    ax_cdf = fig.add_subplot(inner[1], sharex=ax_hist)
    ax_resp = ax_hist.twinx()
    ax_hist.tick_params(labelbottom=False)
    # Plot hist and curves
    plot_climate_hist(ax_hist, clim, bin_edges(beta["tas_bin"]))
    plot_response(ax_resp, beta)
    plot_cdf(ax_cdf, cdf)
    # Labels
    ax_cdf.set_xlabel("Tas [°C]")
    if show_left_labels:
        ax_hist.set_ylabel("Days per Month")
        ax_cdf.set_ylabel("Cum. effect\n[Deaths/100k]")
    if show_right_labels:
        ax_resp.set_ylabel("Daily Response 65+\n[deaths/100k]")
    ax_hist.set_title(title or str(region))

    return ax_hist, ax_resp, ax_cdf

In [20]:
def merge_regions(ds, socioec_ds, forecast_ds):
    # Merge betas and socioeconomics
    if isinstance(ds, xr.DataTree):
        ds = ds.to_dataset()
    merged = xr.merge([
        ds,
        socioec_ds[['pop']],
        forecast_ds[['forecast_mean', 'forecast_std']]
    ], join='inner')

    return merged

def filter_regions(merged, min_population, min_dist_from_mmt, max_dist_from_mmt):
    """
     select regions where population >= min_population and
    |forecast_mean - mmt| <= max_dist_from_mmt.

    mmt in ds has dims (sample, age_cohort, region) — collapsed across sample/age_cohort
    (mean) to get a single per-region MMT for the distance filter.
    """
    mmt_per_region = merged['mmt'].mean(dim=['sample', 'age_cohort'])
    dist_from_mmt = np.abs(merged['forecast_mean'] - mmt_per_region)

    region_mask = (merged['pop'] >= min_population) & (dist_from_mmt <= max_dist_from_mmt)& (dist_from_mmt >= min_dist_from_mmt)
    selected_regions = merged['region'].values[region_mask.values]

    return merged.sel(region=selected_regions)

In [ ]:
### TEST DATA ###
# IR Climate Data
forecast = xr.open_zarr(TAS_FORECAST_URI) - 273.15
# Gridded to Impact Region
forecast_ir = isku_utils.grid_to_ir(forecast, savefile=None)

forecast_ds = forecast_ir.groupby("time.month").mean()
forecast_ds = forecast_ds.assign(
    forecast_mean=forecast_ds['tas'].mean(dim=['number']),
    forecast_std=forecast_ds['tas'].std(dim=['number'])
)

era5 = (
    xr.open_zarr(ERA5_URI).sel(
        time=slice("1996-01-01", "2025-12-31")
    )
    - 273.15
)
era5_ir = isku_utils.grid_to_ir(era5, savefile=None)
era5_ir = era5_ir.assign_coords(month=era5_ir["time"].dt.month)

betas_mmt = effect['/fixed_beta']

/home/emily_zuetell/projects/poreallas/utils/isku_utils.py:28: RuntimeWarning: Failed to open Zarr store with consolidated metadata, but successfully read with non-consolidated metadata. This is typically much slower for opening a dataset. To silence this warning, consider:
1. Consolidating metadata in this existing store with zarr.consolidate_metadata().
2. Explicitly setting consolidated=False, to avoid trying to read consolidate metadata, or
3. Explicitly setting consolidated=True, to raise an error in this case instead of falling back to try reading non-consolidated metadata.
  _region_weights = xr.load_dataset(uri)[
/home/emily_zuetell/projects/poreallas/.venv/lib/python3.14/site-packages/zarr/core/group.py:3413: ZarrUserWarning: Object at zarr.json:Zone.Identifier is not recognized as a component of a Zarr hierarchy.
  warnings.warn(
/home/emily_zuetell/projects/poreallas/utils/isku_utils.py:28: RuntimeWarning: Failed to open Zarr store with consolidated metadata, but successfull

In [22]:
#Select single month
month = 1

month_mask = era5_ir["tas"].month.isin([month]).compute()
reanalysis_slice = era5_ir["tas"].where(month_mask, drop=True)
month_mask = forecast_ir["tas"].sel(time=forecast_ir["tas"].time.dt.month == month).compute()
forecast_slice = forecast_ir["tas"].where(month_mask, drop=True)
# Load Response Curve
betas = betas_mmt["beta_hotonly"] if config.hotonly == "hotonly" else betas_mmt["beta"]
betas_sample = betas.mean(dim = 'sample')
# Load MMT
ref_vals = betas_mmt["mmt"].sel(age_cohort="age65plus")
ref_vals = ref_vals.mean(dim = 'sample')

# Set MMT as median bin when response curve is flat
min_val = betas_mmt["beta"].min(dim="tas_bin")
is_min = betas_mmt["beta"] == min_val
central_tas_bin = betas_mmt["tas_bin"].where(is_min).median(dim="tas_bin", skipna=True)
_polygons_mmt = analysis_utils.xarray_to_gpd(
    central_tas_bin.sel(age_cohort = 'age65plus').mean(dim="sample"), config.polygons
)

In [23]:
### Set Colors ###
b_color = "#999999"
f_color = "#ffbb6f"
r_color = "#5e4c5f"

In [24]:
def compute_slope_at_forecast(merged):
    """
    Compute d(beta)/d(tas) evaluated at each region's forecast_mean,
    for every sample and age_cohort.
    """
    tas_bin = merged['tas_bin'].values

    def _slope_at_point(beta_curve, x0):
        # beta_curve: 1D array over tas_bin; x0: scalar forecast_mean for this region
        dbeta_dtas = np.gradient(beta_curve, tas_bin)
        return np.interp(x0, tas_bin, dbeta_dtas)

    slope = xr.apply_ufunc(
        _slope_at_point,
        merged['beta'],
        merged['forecast_mean'],
        input_core_dims=[['tas_bin'], []],
        output_core_dims=[[]],
        vectorize=True,
        dask='parallelized',
        output_dtypes=[merged['beta'].dtype],
    )

    return slope.rename('beta_slope_at_forecast')

In [25]:
# Load Betas
ds1 = effect['/fixed_beta']
ds1 = merge_regions(ds1, config.socioeconomics, forecast_ds).compute()
# Filter regions w/ 0 pop
ds1 = ds1.where(ds1['pop'] > 1, drop=True)

In [26]:
# Set MMT as median bin when response curve is flat
min_val = betas_mmt["beta"].min(dim="tas_bin")
is_min = betas_mmt["beta"] == min_val
central_tas_bin = betas_mmt["tas_bin"].where(is_min).median(dim="tas_bin", skipna=True)
# Compute distance to MMT for ERA5 (30-yr avg) and the forecast
diff = era5_ir["tas"].groupby("time.month").mean() - central_tas_bin.mean(dim="sample").sel(age_cohort = 'age65plus')
diff_forecast = (
    forecast_ir["tas"].groupby("time.month").mean().mean(dim = 'number')
    - central_tas_bin.mean(dim="sample").sel(age_cohort = 'age65plus')
)

In [ ]:
fig = plt.figure(figsize=(20, 5.5))
gs = fig.add_gridspec(1, 3, width_ratios=[0.6, 1, 1], wspace=0.15)

ax_map2 = fig.add_subplot(gs[1], projection=ccrs.Robinson())
ax_map3 = fig.add_subplot(gs[2], projection=ccrs.Robinson())

robinson = ccrs.Robinson()

# Forecast Map
_polygons_fc_all = analysis_utils.xarray_to_gpd(diff_forecast.mean(dim='month'), config.polygons)

# Slope Map
slope = compute_slope_at_forecast(ds1.sel(age_cohort = 'age65plus').mean(dim = 'month'))
_polygons_slope = analysis_utils.xarray_to_gpd(
    slope.mean(dim="sample"), config.polygons
)

# (ax, gdf, column, cmap name, vmin, vmax, n_colors, colorbar label)
map_specs = [
    (ax_map2, _polygons_fc_all, "value", "bwr", -15, 15, 13, "Forecast-MMT [C]"),
    (ax_map3, _polygons_slope, "beta_slope_at_forecast", "coolwarm", -1, 1, 13,
     "Slope @ Avg. Forecast [Rate/C]"),
]

for a, gdf, column, cmap_name, vmin, vmax, n_colors, cbar_label in map_specs:
    cmap = plt.get_cmap(cmap_name)
    levels = np.linspace(vmin, vmax, n_colors + 1)
    norm = mcolors.BoundaryNorm(levels, cmap.N, extend="both")

    ## Mask Out Caspian Sea ##
    gdf_rob = gdf.to_crs(robinson.proj4_init)

    gdf_rob.plot(
        ax=a, column=column, cmap=cmap, norm=norm, linewidth=0,
        aspect=None, missing_kwds={"color": "lightgray"},
    )
    a.coastlines(linewidth=0.6, color="k")
    a.set_extent([-180, 180, -60, 90], crs=ccrs.PlateCarree())
    a.spines["geo"].set_visible(False)
    fig.colorbar(plt.cm.ScalarMappable(norm=norm, cmap=cmap), ax=a,
                 orientation="horizontal", shrink=0.7, aspect=30, pad=0.03,
                 label=cbar_label)
# Plot Region Panel
ax_hist, ax_resp, _ = plot_region_panel(fig, gs[0], "BWA.3", title="Kgalagadi, Botswana",
                                        show_left_labels=True, show_right_labels=True)

h1, l1 = ax_hist.get_legend_handles_labels()
h2, l2 = ax_resp.get_legend_handles_labels()
ax_resp.legend(h1 + h2, l1 + l2, loc="upper left", fontsize=8, frameon=False)

fig.savefig(f"{config.version}_method_diagram_map.png", dpi=600, bbox_inches="tight")